# 03. Song-Level Dataset

1. Data Loading
2. Song-Level Aggregation
   - 2.1 Metadata Consistency Check
   - 2.2 Chart History per Song
   - 2.3 Song Table Construction
3. Release Year Correction
   - 3.1 Correction Rule
   - 3.2 Release Year Range Check
4. Summary
5. Export

## 1. Data Loading

### Objective
Load the cleaned chart-level records from `02_data_cleaning.ipynb`.

In [1]:
from pathlib import Path

import pandas as pd

INTERIM_DIR = Path("../data/interim")

df = pd.read_pickle(INTERIM_DIR / "chart_records_clean.pkl")

print(f"Chart records: {len(df):,}")
print(f"Unique songs (song_key): {df['song_key'].nunique():,}")

Chart records: 25,167
Unique songs (song_key): 7,988


## 2. Song-Level Aggregation

### Objective
Convert chart-level records into one row per song so that songs with long chart runs are not counted repeatedly.

### 2.1 Metadata Consistency Check

#### Objective
Check whether records of the same song share the same metadata and lyrics before aggregation.

#### Procedure
- For each `song_key`, count distinct values of song name, artist, release date, genre, and lyrics.
- Report songs with more than one distinct value.
- The song table uses the values from the song's first chart appearance (2.3).

In [2]:
df["lyrics_tuple"] = df["lyrics"].apply(tuple)

check_columns = ["song_name", "artist", "release_date", "genre", "lyrics_tuple"]

n_distinct = df.groupby("song_key")[check_columns].nunique(dropna=False)
inconsistent = (n_distinct > 1).sum()

display(pd.DataFrame({
    "songs with >1 value": inconsistent,
    "share (%)": (inconsistent / len(n_distinct) * 100).round(2),
}))

lyric_variants = n_distinct.index[n_distinct["lyrics_tuple"] > 1]
if len(lyric_variants):
    print("\nSample songs with multiple lyric versions (first month of each version):")
    display(
        df[df["song_key"].isin(lyric_variants[:5])]
        .sort_values(["song_key", "chart_year", "chart_month"])
        .drop_duplicates(["song_key", "lyrics_tuple"])
        [["song_key", "song_name", "artist", "chart_year", "chart_month", "lyric_line_count"]]
    )

,songs with >1 value,share (%)
song_name,0,0.0
artist,0,0.0
release_date,0,0.0
genre,0,0.0
lyrics_tuple,0,0.0


### 2.2 Chart History per Song

#### Objective
Summarize each song's chart history.

#### Procedure
- `first_chart_year`, `first_chart_month`: first chart appearance
- `last_chart_year`, `last_chart_month`: last chart appearance
- `n_chart_months`: number of chart months
- `peak_rank`: highest chart rank

In [3]:
df["chart_ym"] = df["chart_year"] * 100 + df["chart_month"]

history = df.groupby("song_key").agg(
    first_ym=("chart_ym", "min"),
    last_ym=("chart_ym", "max"),
    n_chart_months=("chart_ym", "size"),
    peak_rank=("chart_rank", "min"),
)

history["first_chart_year"] = history["first_ym"] // 100
history["first_chart_month"] = history["first_ym"] % 100
history["last_chart_year"] = history["last_ym"] // 100
history["last_chart_month"] = history["last_ym"] % 100
history = history.drop(columns=["first_ym", "last_ym"])

display(history["n_chart_months"].describe())

print("\nSongs by number of chart months:")
display(
    pd.cut(history["n_chart_months"], bins=[0, 1, 3, 6, 12, 1000],
           labels=["1", "2–3", "4–6", "7–12", "13+"])
    .value_counts().sort_index().to_frame("songs")
)

count    7988.000000
mean        3.150601
std         3.281345
min         1.000000
25%         1.000000
50%         2.000000
75%         4.000000
max        80.000000
Name: n_chart_months, dtype: float64


Songs by number of chart months:


,songs
n_chart_months,
1,2698
2–3,2855
4–6,1800
7–12,477
13+,158


### 2.3 Song Table Construction

#### Objective
Build one row per song.

#### Procedure
- Take metadata and lyrics from the first chart appearance (earliest month, then highest rank).
- Merge the chart history from 2.2.

In [4]:
first_appearance = (
    df.sort_values(["chart_year", "chart_month", "chart_rank"])
      .drop_duplicates("song_key", keep="first")
      .set_index("song_key")
)

meta_columns = [
    "key_source", "song_id", "song_name", "artist", "album", "genre",
    "release_date", "release_date_parsed", "lyrics", "lyric_line_count",
]

songs = first_appearance[meta_columns].join(history).reset_index()

print(f"Songs: {len(songs):,}")
display(songs.drop(columns=["lyrics"]).head())

Songs: 7,988


,song_key,key_source,song_id,song_name,artist,album,genre,release_date,release_date_parsed,lyric_line_count,n_chart_months,peak_rank,first_chart_year,first_chart_month,last_chart_year,last_chart_month
0,38541,song_id,38541,부메랑,김건모,Growing,발라드,1999.11.20,1999-11-20,42,2,1,2000,1,2000,2
1,85257,song_id,85257,바꿔,이정현,Let's Go To My Star,댄스,1999.10,1999-10-01,39,8,2,2000,1,2000,8
2,91324,song_id,91324,사랑해 그리고 기억해,god,Chapter 2,발라드,1999.11.25,1999-11-25,62,4,3,2000,1,2000,9
3,85674,song_id,85674,영원,스카이,Final Fantasy,"발라드, 록/메탈",1999.11.16,1999-11-16,38,3,1,2000,1,2000,3
4,91320,song_id,91320,실연,코요태,실연,댄스,1999.11.06,1999-11-06,60,1,5,2000,1,2000,1


## 3. Release Year Correction

### Objective
Define a reliable song-level time variable (`release_year`) for temporal analysis (RQ2).

### 3.1 Correction Rule

#### Procedure
- Default: `release_year` = year of `release_date`.
- Replace with `first_chart_year` when:
  - the release date is missing or cannot be parsed, or
  - the release date is later than the end of the first chart month (likely a re-release date).
- `release_year_source` records which rule was applied.

In [5]:
first_month_end = (
    pd.to_datetime(dict(year=songs["first_chart_year"], month=songs["first_chart_month"], day=1))
    + pd.offsets.MonthEnd(0)
)

is_missing = songs["release_date_parsed"].isna()
is_after_chart = songs["release_date_parsed"] > first_month_end

songs["release_year_raw"] = songs["release_date_parsed"].dt.year.astype("Int64")
songs["release_year"] = songs["release_year_raw"]
songs.loc[is_missing | is_after_chart, "release_year"] = songs.loc[is_missing | is_after_chart, "first_chart_year"]

songs["release_year_source"] = "release_date"
songs.loc[is_after_chart, "release_year_source"] = "first_chart (release after chart)"
songs.loc[is_missing, "release_year_source"] = "first_chart (release missing)"

display(songs["release_year_source"].value_counts().to_frame("songs"))

print("\nCorrected songs:")
display(
    songs[songs["release_year_source"] != "release_date"]
    [["song_name", "artist", "release_date", "first_chart_year", "first_chart_month",
      "release_year", "release_year_source"]]
    .sort_values(["first_chart_year", "first_chart_month"])
    .head(20)
)

,songs
release_year_source,
release_date,7896
first_chart (release after chart),89
first_chart (release missing),3



Corrected songs:


,song_name,artist,release_date,first_chart_year,first_chart_month,release_year,release_year_source
9,비젼,유승준,2001.09.01,2000,1,2000,first_chart (release after chart)
23,다 알아요,양파,2003.11.13,2000,1,2000,first_chart (release after chart)
28,고요한 밤 거룩한 밤,조성모,2002.01,2000,1,2000,first_chart (release after chart)
33,연가,유승준,2001.09.01,2000,1,2000,first_chart (release after chart)
41,애수(哀愁),이문세,2002.02.09,2000,1,2000,first_chart (release after chart)
51,슬픈 사랑의 노래 (WITH 이소라),이문세,2002.02.09,2000,1,2000,first_chart (release after chart)
128,그대가 그대를,이승환,2003.03.12,2000,5,2000,first_chart (release after chart)
191,OP.T (Operation Takeover),H.O.T.,2010.07.15,2000,8,2000,first_chart (release after chart)
214,성녀,이상은 (=Lee-tzsche),2012.08.21,2000,10,2000,first_chart (release after chart)
239,찾길 바래,유승준,2001.09.01,2000,12,2000,first_chart (release after chart)


### 3.2 Release Year Range Check

#### Objective
Check songs whose release year is far from their first chart appearance.

#### Procedure
- `chart_lag` = `first_chart_year` − `release_year`.
- Report the distribution of `chart_lag` and songs released before 2000.
- No records are changed in this step; old songs that re-entered the chart (e.g. revivals) are kept with their original release year.

In [6]:
songs["chart_lag"] = songs["first_chart_year"] - songs["release_year"]

print("Chart lag (years):")
display(
    pd.cut(songs["chart_lag"], bins=[-1, 0, 1, 2, 5, 100],
           labels=["0", "1", "2", "3–5", "6+"])
    .value_counts().sort_index().to_frame("songs")
)

print(f"\nSongs released before 2000: {(songs['release_year'] < 2000).sum():,}")
display(
    songs[songs["chart_lag"] >= 6]
    [["song_name", "artist", "release_date", "first_chart_year", "chart_lag", "n_chart_months"]]
    .sort_values("chart_lag", ascending=False)
    .head(20)
)

Chart lag (years):


,songs
chart_lag,
0,7408
1,477
2,23
3–5,40
6+,40



Songs released before 2000: 113


,song_name,artist,release_date,first_chart_year,chart_lag,n_chart_months
5714,잠 못 드는 밤 비는 내리고,김건모,1992.10.29,2015,23,1
1017,Sweet Dreams (Are Made Of This),Eurythmics,1983,2004,21,1
5703,나 어릴 적 꿈,터보,1995.07.01,2015,20,1
5692,잘못된 만남,김건모,1995.01.24,2015,20,3
6751,빛 (Hope),H.O.T.,1998.09.01,2018,20,1
6173,커플,젝스키스,1998.10.03,2016,18,1
5695,그녀와의 이별,김현정,1997.01.01,2015,18,2
5700,말해줘,지누션,1997.03.01,2015,18,1
5706,초대,엄정화,1998.07,2015,17,1
5710,너를 사랑해,S.E.S.,1998.11,2015,17,1


## 4. Summary

In [7]:
print(f"Songs: {len(songs):,}")
print(f"Songs with corrected release year: {(songs['release_year_source'] != 'release_date').sum():,}")

summary = pd.DataFrame({
    "songs (release_year)": songs.groupby("release_year").size(),
    "songs (first_chart_year)": songs.groupby("first_chart_year").size(),
}).fillna(0).astype(int)

display(summary)

Songs: 7,988
Songs with corrected release year: 92


,songs (release_year),songs (first_chart_year)
1983,1,0
1988,1,0
1989,4,0
1990,1,0
1991,2,0
1992,1,0
1995,4,0
1996,1,0
1997,5,0
1998,8,0


## 5. Export

### Output
- `data/interim/songs.pkl`: one row per song (lyrics kept as lists)

In [8]:
song_columns = [
    "song_key", "key_source", "song_id", "song_name", "artist", "album", "genre",
    "release_date", "release_year_raw", "release_year", "release_year_source",
    "first_chart_year", "first_chart_month", "last_chart_year", "last_chart_month",
    "n_chart_months", "peak_rank", "chart_lag",
    "lyrics", "lyric_line_count",
]

songs[song_columns].to_pickle(INTERIM_DIR / "songs.pkl")

print(f"Saved {len(songs):,} songs to {INTERIM_DIR / 'songs.pkl'}")

Saved 7,988 songs to ../data/interim/songs.pkl
